# Collect GVGAI ASCII transitions on Google Colab

End-to-end Colab notebook for collecting Aliens / Chopper / Waves ASCII transitions using GVGAI's `sampleMCTS` agent, writing shards in the layout `world_model/train_ascii_vae.py::AllAsciiFramesDataset` reads, and uploading the result as a single tar to Google Drive for the training notebook (`notebooks/train_ascii_vae_colab.ipynb`) to pick up.

## Runtime

1. `Runtime` → `Change runtime type` → **CPU** (no GPU needed; MCTS + Java are CPU-bound).
2. `Runtime` → `Change runtime type` → enable **High-RAM** if offered (optional).

At 500k frames × 3 games × 40 ms/tick MCTS this typically takes 2–4 hours on a Colab CPU runtime. Free-tier sessions last up to ~12 hours, so it fits comfortably. If the session dies mid-run, just re-run the full-collection cell — the Java collector resumes at the next free `shard_NNNNN/`.

## Prereqs

1. **Push your branch first.** This notebook clones `REPO_BRANCH` from GitHub; any local-only commits won't be visible to Colab.
2. Your Drive must have ~1 GB free under `MyDrive/DecoupliWo/` (the final tar is ~750 MB at the default 500k-frames-per-game target).

## 1. Verify runtime

In [ ]:
!uname -a && nproc && free -h

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths (edit here if yours differ)

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/wessimpson/DecoupliWo.git'
REPO_BRANCH = 'gameNGen-world_model-v2'
REPO_DIR = Path('/content/DecoupliWo')
GVGAI_DIR = REPO_DIR / 'gvgai'
GVGAI_BUILD = GVGAI_DIR / 'out' / 'production' / 'gvgai'
LOCAL_DATA_DIR = REPO_DIR / 'data' / 'transitions'
LOCAL_TAR = Path('/content/transitions.tar')

DRIVE_ROOT = Path('/content/drive/MyDrive/DecoupliWo')
DRIVE_TAR = DRIVE_ROOT / 'transitions.tar'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('REPO_DIR      =', REPO_DIR)
print('LOCAL_DATA_DIR=', LOCAL_DATA_DIR)
print('DRIVE_TAR     =', DRIVE_TAR)

## 4. Install a JDK

Colab runtimes don't ship with `javac`. `default-jdk` is typically Temurin 21 and takes ~30 s to install.

In [ ]:
!apt-get -qq update && apt-get -qq install -y default-jdk > /dev/null
!java -version
!javac -version

## 5. Clone the DecoupliWo repo

In [ ]:
import subprocess

if REPO_DIR.is_dir() and (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

## 6. Clone GVGAI into `gvgai/`

The DecoupliWo repo references `gvgai/` as a submodule pinned to a private fork commit. Upstream `GAIGResearch/GVGAI` master has everything this collector depends on (`core.game.*`, `core.vgdl.*`, `ontology.Types`, `tracks.ArcadeMachine`, `tracks.singlePlayer.advanced.sampleMCTS`), so we just clone upstream.

In [ ]:
if GVGAI_DIR.is_dir() and (GVGAI_DIR / 'src').is_dir():
    print(f'GVGAI already present at {GVGAI_DIR}')
else:
    import shutil
    if GVGAI_DIR.exists():
        shutil.rmtree(GVGAI_DIR)
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/GAIGResearch/GVGAI.git', str(GVGAI_DIR)],
        check=True,
    )

!ls {GVGAI_DIR}/src/tracks/singlePlayer/advanced/sampleMCTS/

## 7. Install the collector sources and compile GVGAI

Copies the Java stubs from `gvgai_java_stubs/src/tracks/singlePlayer/ascii/` into `gvgai/src/tracks/singlePlayer/ascii/` (alongside the existing GVGAI sources) and compiles everything to `gvgai/out/production/gvgai/`. This takes ~1 min.

In [ ]:
import shutil

ascii_src_dir = GVGAI_DIR / 'src' / 'tracks' / 'singlePlayer' / 'ascii'
ascii_src_dir.mkdir(parents=True, exist_ok=True)
for p in (REPO_DIR / 'gvgai_java_stubs' / 'src' / 'tracks' / 'singlePlayer' / 'ascii').glob('*.java'):
    shutil.copy2(p, ascii_src_dir / p.name)
print('copied:', sorted(p.name for p in ascii_src_dir.glob('*.java')))

GVGAI_BUILD.mkdir(parents=True, exist_ok=True)
!cd {GVGAI_DIR} && javac -d {GVGAI_BUILD} $(find src -name '*.java')
!ls {GVGAI_BUILD}/tracks/singlePlayer/ascii/

## 8. Smoke test (~1 min)

Before the multi-hour full run, verify each game's mapping JSON resolves to real VGDL sprite types by collecting a handful of frames at a cheap 10 ms/tick MCTS budget and eyeballing frame 0.

In [ ]:
!cd {REPO_DIR} && python -m data.collect_gvgai_ascii \
    --games aliens,chopper,waves \
    --frames-per-game 2000 \
    --mcts-ms 10 \
    --chunk-size 500

In [ ]:
import numpy as np

for game in ('aliens', 'chopper', 'waves'):
    path = LOCAL_DATA_DIR / 'train' / game / 'shard_00000' / 'obs.npy'
    arr = np.load(path)
    chars = sorted({chr(c) for c in np.unique(arr)})
    print(f'=== {game}: shape={arr.shape} dtype={arr.dtype} chars={chars} ===')
    for row in arr[0]:
        print(''.join(chr(c) for c in row))
    print()

## 9. Full collection (~2–4 hrs)

Wipes the smoke-test shards first so they don't pollute the real dataset. If the runtime dies mid-collection, just re-run this cell — shard indices resume from the next free slot.

Expected output footprint: roughly 500 shards total, ~750 MB in `LOCAL_DATA_DIR`.

In [ ]:
import shutil

if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)

!cd {REPO_DIR} && python -m data.collect_gvgai_ascii \
    --games aliens,chopper,waves \
    --frames-per-game 500000 \
    --mcts-ms 40 \
    --chunk-size 5000

## 10. Summarize local dataset

In [ ]:
import numpy as np

for split in ('train', 'test'):
    split_root = LOCAL_DATA_DIR / split
    if not split_root.is_dir():
        continue
    for game_dir in sorted(split_root.iterdir()):
        shards = sorted(game_dir.glob('shard_*'))
        frames = sum(int(np.load(s / 'obs.npy', mmap_mode='r').shape[0]) for s in shards)
        print(f'{split}/{game_dir.name}: {len(shards)} shards, {frames:,} frames')

bytes_used = int(subprocess.check_output(['du', '-sb', str(LOCAL_DATA_DIR)]).split()[0])
print(f'\ntotal on disk: {bytes_used / 1e6:.1f} MB')

## 11. Tar and upload to Drive

Tars `data/transitions/` to `/content/transitions.tar` locally (fast — local SSD), then copies the single tar to Drive (a few minutes for ~750 MB). This matches the exact filename the training notebook's cell 6 looks for.

In [ ]:
if LOCAL_TAR.exists():
    LOCAL_TAR.unlink()

!cd {REPO_DIR} && tar -cf {LOCAL_TAR} -C data transitions
print(f'local tar: {LOCAL_TAR}  ({LOCAL_TAR.stat().st_size / 1e9:.2f} GB)')

import shutil
shutil.copyfile(LOCAL_TAR, DRIVE_TAR)
print(f'copied to: {DRIVE_TAR}  ({DRIVE_TAR.stat().st_size / 1e9:.2f} GB)')

## 12. Verify Drive tar

Opens the Drive tar and lists the first 20 entries + total entry count so you can confirm it contains `transitions/train/*` and `transitions/test/*` before launching the training notebook.

In [ ]:
import tarfile

with tarfile.open(DRIVE_TAR, 'r') as tf:
    names = tf.getnames()
print(f'{len(names):,} entries in {DRIVE_TAR}')
for name in names[:20]:
    print(' ', name)
print('  ...')
print('\nNext step: run `notebooks/train_ascii_vae_colab.ipynb`.')